# Portfolio-opdracht 2: Voorspel verhuuraantallen (CityBike)

<a id="top"></a>

|  |  |
|---|---|
| **Teamnaam (Kaggle)** | ALEX |
| **Competitie** | [Voorspel verhuuraantallen 2026](https://www.kaggle.com/competitions/voorspel-verhuuraantallen-2026) |
| **Datum** | 29-11-2026 |
| **Versie** | v1.0 |

| Teamlid | Studentnummer | Kaggle-gebruikersnaam |
|---|---|---|
| Thomas Schooneman | 25056492 | Thomas-004 |
| Levi Postma | 25042157 | Ikkekoe |
| Alec Veld | 25064185 | Alec Veld |

## Inhoudsopgave

1. [Setup, configuratie en data inladen](#h1)
2. [Opdracht 1: Exploratieve Data Analyse](#h2)
3. [Opdracht 2: Time series feature engineering](#h3)
4. [Opdracht 3: Modelleren](#h4)
5. [Bevindingen](#h5)
6. [Opdracht 4: Conclusie en aanbevelingen](#h6)
7. [Referenties en GenAI-gebruik](#h7)
8. [Inleverchecklist](#h8)

<a id="h1"></a>
# 1. Setup, configuratie en data inladen

[Terug naar boven](#top)

## 1.1 Imports

In [49]:
# Standaardbibliotheken
import warnings
from dataclasses import dataclass, field, replace
from datetime import datetime, date
from pathlib import Path
from html import escape

# Data en visualisatie
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

# Statistiek en tijdreeksen
from scipy import signal, stats
from scipy.fft import rfft, rfftfreq
from scipy.stats import loguniform, randint
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

# Machine learning
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (HistGradientBoostingRegressor,
                              RandomForestRegressor)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 1.2 Configuratie

Alle instellingen staan op één plek, zodat het notebook reproduceerbaar is
(vaste `random_state`) en paden/bestandsnamen makkelijk aan te passen zijn.

In [50]:
# Opmaak voor figuren
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)

TARGET = "cnt"

## 1.3 Gen-AI gebruik logboek

In [51]:
AI_LOG = []

def log_ai(titel, model, link, waar, datum=None):
    """Voeg een AI-chat toe aan het logboek.

    titel : titel van de chat
    model : model + versie, bv. "Claude Sonnet 4.5"
    link  : link naar het gesprek
    waar  : waar gebruikt, bv. "§3.2, cel 14"
    datum : standaard vandaag, anders bv. "2026-11-24"
    """
    entry = {
        "Titel van de chat": titel,
        "Model + versie": model,
        "Datum": str(datum or date.today()),
        "Link naar gesprek": link,
        "Waar gebruikt (§ / cel)": waar,
    }
    # Voorkomt dubbele regels als je een cel opnieuw draait
    if entry not in AI_LOG:
        AI_LOG.append(entry)

def show_ai_log():
    """Toon het logboek als tabel met klikbare links."""
    if not AI_LOG:
        print("Nog geen AI-gebruik gelogd.")
        return
    df = pd.DataFrame(AI_LOG)
    df.insert(0, "Prompt nr.", range(1, len(df) + 1))
    df["Link naar gesprek"] = df["Link naar gesprek"].apply(
        lambda u: f'<a href="{escape(u)}" target="_blank">link</a>'
    )
    display(df.style.hide(axis="index"))

# Gebruik

# log_ai(
#     titel="____",
#     model="____",
#     link="____",
#     waar="____"
# )

## 1.4 Data inladen

In [52]:
sample_submission = pd.read_csv("sample_submission.csv")
test_raw = pd.read_csv("test.csv")
train_raw = pd.read_csv("train.csv")

<a id="h2"></a>
# 2. Opdracht 1: Exploratieve Data Analyse

[Terug naar boven](#top)

## 2.1 Datatypes en basisstatistieken

In [58]:
def describe_columns(df):
    """Geef per kolom datatype, ontbrekende waarden en basisstatistieken."""
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    numeric = df.select_dtypes("number").describe().T
    return summary.join(numeric[["mean", "std", "min", "50%", "max"]])

datasets = {"sample_submission" : sample_submission, "test_raw" : test_raw, "train_raw" : train_raw}

for i, j in datasets.items():
    display(i, describe_columns(j), j.shape)

assert set(test_raw) == set(train_raw) - {TARGET}

'sample_submission'

,dtype,n_missing,pct_missing,n_unique,mean,std,min,50%,max
date_hour,object,0,0.0,456,NaN,NaN,NaN,NaN,NaN
cnt,int64,0,0.0,349,413.927632,231.532822,0.0,421.0,799.0


(456, 2)

'test_raw'

,dtype,n_missing,pct_missing,n_unique,mean,std,min,50%,max
date_hour,object,0,0.0,456,NaN,NaN,NaN,NaN,NaN
holiday,int64,0,0.0,1,0.000000,0.000000,0.0000,0.0000,0.0000
weathersit,int64,0,0.0,3,1.592105,0.607882,1.0000,2.0000,3.0000
temp,float64,0,0.0,21,0.360132,0.079432,0.1600,0.3600,0.6000
atemp,float64,0,0.0,24,0.363691,0.076354,0.1818,0.3636,0.6212
hum,float64,0,0.0,49,0.714868,0.177732,0.3300,0.7500,1.0000
windspeed,float64,0,0.0,17,0.157639,0.106499,0.0000,0.1343,0.4925


(456, 7)

'train_raw'

,dtype,n_missing,pct_missing,n_unique,mean,std,min,50%,max
date_hour,object,0,0.0,16637,NaN,NaN,NaN,NaN,NaN
holiday,int64,0,0.0,2,0.028671,0.166885,0.00,0.000,1.0000
weathersit,int64,0,0.0,4,1.415580,0.637298,1.00,1.000,4.0000
temp,float64,0,0.0,50,0.504745,0.192369,0.02,0.520,1.0000
atemp,float64,0,0.0,65,0.482608,0.171557,0.00,0.500,1.0000
hum,float64,0,0.0,89,0.624756,0.193227,0.00,0.620,1.0000
windspeed,float64,0,0.0,30,0.190310,0.121915,0.00,0.194,0.8507
cnt,int64,0,0.0,867,190.477009,182.026755,1.00,143.000,977.0000


(16637, 8)

## 2.2 Datakwaliteit en opschoning

Controleer onder meer: dubbele tijdstippen, ontbrekende uren in de reeks,
ontbrekende waarden, onmogelijke waarden (buiten 0-1) en verdachte waarden
(bijv. luchtvochtigheid van precies 0).